In [4]:
import re
import logging

In [5]:
logging.basicConfig(level=logging.INFO)

In [ ]:
# 由于plog有最大1024的长度限制，因此需要先合并拆分多行的tiling trace日志
def is_complete_tiling_log(current_kernel, kernel_name):
    # return content.endswith('.')
    return current_kernel != kernel_name

def assemble_lines(current_kernel, current_lines):
    prefix = f'[KernelTrace][{current_kernel}]'
    return prefix + ''.join(current_lines)

#todo 需要整改这里tiling data 结束判断条件一级tiling cache status匹配模式
def parse_tiling_logs(file_path):
    tiling_logs = []
    current_lines = []
    current_kernel = None
    kernel_match_pattern = re.compile(
        r'\[KernelTrace\]\[(?P<full_name>(?P<kernel_name>Tiling|CacheableTiling)_(?P<op_name>.+)_(?P<index>\d+))\](?P<content>.*)')

    with open(file_path, 'r', encoding='utf-8') as fin:
        for line in fin:
            match = kernel_match_pattern.search(line)
            # 只有Tiling的Trace日志走下面流程, 因此保证kernel_name符合要求
            if not match:
                continue
            
            kernel_name = match.group('full_name')
            if current_kernel is None and len(current_lines) == 0:
                current_kernel = kernel_name
                logging.debug(f'Processing {current_kernel}, {match.groupdict()}')
             
             # 完整日志则直接保存，这里要恢复下KernelTrace前缀
            if is_complete_tiling_log(current_kernel, kernel_name):
                tiling_logs.append(assemble_lines(current_kernel, current_lines))
                current_kernel = kernel_name
                current_lines = []

            content = match.group('content')
            current_lines.append(content)

        # 处理尾块
        if current_lines:
            tiling_logs.append(assemble_lines(current_kernel, current_lines))
            current_lines = []
            current_kernel = None

    return tiling_logs


# unittest
file_path = 'example_log.txt'  # 替换为你的日志文件路径
tiling_entries = parse_tiling_logs(file_path)
for entry in tiling_entries:
    print("Tiling Log:")
    print(entry)
    print("=" * 80)

In [23]:
# pattern = re.compile(
#     r'\[KernelTrace\]\[(?P<kernel>[^_\]]+)_(?P<op>[^_\]]+)_(?P<index>\d+)\]'
#     r'Tiling key is (?P<key>\d+), atomic clean flag is (?P<flag>\d+), block_dim is (?P<dim>\d+), tiling_cond is (?P<cond>-?\d+),'
#     r'\s*tilingData size: (?P<size>\d+) bytes,\s*TilingData:\s*(?P<data>(?:0x[0-9a-fA-F]+\s*)+).*?'
#     r'Tiling cache (?:status: (?P<status>missed|hit)|is (?P<status2>disabled))\.'
# )

def parse_single_tiling_log(line):
    result = {}

    # 提取 Kernel 信息（如 Tiling_Add_1670）, 暂时不区分KernelName和OpName
    kernel_match = re.search(r'\[KernelTrace\]\[([^\]]+)\]', line)
    if kernel_match:
        result["kernel_op_name"] = kernel_match.group(1)

    # 提取 Tiling key
    key_match = re.search(r'Tiling key is (\d+)', line)
    if key_match:
        result["tiling_key"] = int(key_match.group(1))

    # 提取 atomic clean flag
    atomic_flag_match = re.search(r'atomic clean flag is (\d+)', line)
    if atomic_flag_match:
        result["atomic_clean_flag"] = int(atomic_flag_match.group(1))

    # 提取 block dim
    block_dim_match = re.search(r'block_dim is (\d+)', line)
    if block_dim_match:
        result["block_dim"] = int(block_dim_match.group(1))

    # 提取 tiling_cond
    tiling_cond_match = re.search(r'tiling_cond is (-?\d+)', line)
    if tiling_cond_match:
        result["tiling_cond"] = int(tiling_cond_match.group(1))

    # 提取 tilingData size
    size_match = re.search(r'tilingData size: (\d+) bytes', line)
    if size_match:
        result["tiling_data_size"] = int(size_match.group(1))

    # 提取 TilingData
    tiling_data_match = re.search(r'TilingData:\s*((?:0x[0-9a-fA-F]{2}\s*)+)', line)
    if tiling_data_match:
        result["tiling_data"] = tiling_data_match.group(1).strip()

    # 提取 cache 状态, 记一个遗留问题, 后面改下trace日志格式
    status_match = re.search(r'Tiling cache status: (?P<status>\w+),', line)
    if status_match:
        result["cache_status"] = status_match.group('status')

    return result


In [ ]:
file_path = "example_log.txt"
tiling_logs = parse_tiling_logs(file_path)

for idx, line in enumerate(tiling_logs):
    results = parse_single_tiling_log(line)
    print(f"Tiling Trace {idx + 1}:")
    print(f"  Kernel Name   : {results['kernel_op_name']}")
    print(f"  Tiling Key    : {results['tiling_key']}")
    print(f"  Atomic Flag   : {results['atomic_clean_flag']}")
    print(f"  Block Dim     : {results['block_dim']}")
    print(f"  Tiling Cond   : {results['tiling_cond']}")
    print(f"  Tiling Data Size   : {results['tiling_data_size']}")
    print(f"  Tiling Data   : {results['tiling_data']}")
    print(f"  Cache Status   : {results['cache_status']}")
    print("-" * 80)

In [ ]:
file_path = "example_log.txt"
output_file = "parsed_tiling_output.txt"

tiling_logs = parse_tiling_logs(file_path)

with open(output_file, 'w', encoding='utf-8') as f:
    for idx, line in enumerate(tiling_logs):
        results = parse_single_tiling_log(line)
        f.write(f"Tiling Trace {idx + 1}:\n")
        f.write(f"  Kernel Name       : {results['kernel_op_name']}\n")
        f.write(f"  Tiling Key        : {results['tiling_key']}\n")
        f.write(f"  Atomic Flag       : {results['atomic_clean_flag']}\n")
        f.write(f"  Block Dim         : {results['block_dim']}\n")
        f.write(f"  Tiling Cond       : {results['tiling_cond']}\n")
        f.write(f"  Tiling Data Size  : {results['tiling_data_size']}\n")
        f.write(f"  Tiling Data       : {results['tiling_data']}\n")
        f.write(f"  Cache Status      : {results['cache_status']}\n")
        f.write("\n")

print(f"Save results to {output_file}")